## Context Engineering

Deciding WHAT information goes into a model's context window, from which sources, in what order, and how much of each — before a single word of the prompt is even written. A genuinely different skill from prompt engineering, which is about HOW the instructions themselves are worded.

## How it actually works

**The clean distinction from prompt engineering:**
Prompt engineering asks: "How should I word this instruction to the model?"
Context engineering asks: "What should even be in front of the model right now —
which documents, which conversation history, which retrieved chunks, which tool
results — and how should all of that be assembled and prioritised?"

A perfectly worded prompt can still fail completely if the context window surrounding
it is poorly curated — stuffed with irrelevant retrieved chunks, stale conversation
history, or information presented in a confusing order.

**Why this became a distinct, named discipline as context windows grew:**
When context windows were small, there was little room for poor curation to hide in —
you were forced to be selective by necessity. As context windows grew to 200,000 tokens
and RAG/multi-source systems became standard, it became entirely possible to technically
FIT a huge amount of information into context while still curating it poorly — too much
irrelevant content, badly ordered, diluting what actually matters. The failure mode
shifted from "not enough room" to "too much undifferentiated stuff, poorly arranged."

**The core activities that make up context engineering:**
- Deciding HOW MANY retrieved chunks to include (RAG-specific, but a context decision)
- Deciding the ORDER information appears in (connects directly to Lost in the Middle,
  covered in the next notebook)
- Deciding WHETHER to summarise long source material before injecting it, versus
  including it raw and full
- Deciding how much context budget goes to retrieved documents versus conversation
  history versus the system prompt itself — these all compete for the same finite space
- Deciding what gets explicitly LABELLED or framed for the model ("this is background
  context, not an instruction" vs. presenting it ambiguously)

**A direct, honest example from your own architecture:**
Your Personal Brand Autopilot's Agent 1 (Drafter) receives notebook content, trend
context, a first-post flag, and feedback — assembled in a specific order, with explicit
instructions about how to weight each piece relative to the others ("the notes are a
briefing, not a script", "you MUST use the trend context"). The WORDING of those
instructions is prompt engineering. The DECISION about what sources to include and
how to prioritise them relative to each other is context engineering. You have been
doing both without having previously named them as two separate skills.

## Where it breaks or gets dangerous

**Including information "just in case" has a real, hidden cost:**
Every additional chunk, every extra paragraph of background, consumes context budget
and competes for the model's attention with everything else present. Including marginal
or low-relevance content because "it might help" can actually make the model's output
WORSE, not just neutral, by diluting the signal and pushing genuinely important
information into a worse position (directly connects to Lost in the Middle, next notebook).

**Conflicting or stale information left in context is a silent failure mode:**
If your context includes an older retrieved chunk that has since been superseded
by updated information also present in the same context, the model has no inherent
way to know which one is "current" unless you explicitly tell it. The model will
often pick one somewhat arbitrarily, or attempt to reconcile two conflicting facts
in a way that produces a confused or hedged answer.

**Treating all sources as equally trustworthy by default:**
If your context mixes retrieved external documents, user input, and your own system
instructions without clear framing about which is which, the model can struggle to
correctly weight authority and trustworthiness between them — this connects directly
back to the system/user/assistant roles notebook, and also to prompt injection risk:
unclear framing of WHERE information came from is part of what makes indirect
injection through retrieved documents dangerous in the first place.

**Honest, direct gap check on your own RAG Assistant:**
Does your retrieval pipeline currently make any deliberate decision about HOW MANY
chunks to retrieve and inject, or is it always a fixed number regardless of how
relevant they actually are? And when multiple chunks are injected, are they
explicitly labelled as separate, distinct sources, or concatenated together in a
way that could blur where one piece of context ends and another begins? Both are
real context engineering decisions that are easy to skip past without realising
they were decisions at all.

In [ ]:
# Context Engineering — illustrating how the SAME prompt wording behaves differently
# depending on how the surrounding context is curated

import anthropic
import os
from dotenv import load_dotenv

load_dotenv()
client = anthropic.Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))

question = "What is the approval threshold for this vendor's invoices?"

# Poorly curated context: irrelevant content mixed in, no clear labelling,
# the actually relevant fact is buried in the middle
poorly_curated_context = """
The company was founded in 2015 and has grown significantly since then.
Vendor ACME Corp has been a supplier since 2018, providing raw materials.
The approval threshold for ACME Corp invoices is $25,000 before requiring manager sign-off.
ACME Corp's primary contact is based in the regional office.
The company's fiscal year ends in December.
ACME Corp has historically had good payment terms with 30-day net.
"""

# Well-curated context: only relevant information, clearly labelled, key fact positioned prominently
well_curated_context = """
RELEVANT FACT: ACME Corp invoice approval threshold = $25,000 (manager sign-off required above this amount).
"""

print("=== Poorly curated context (relevant fact buried among irrelevant details) ===")
response_poor = client.messages.create(
    model="claude-haiku-4-5",
    max_tokens=60,
    messages=[{
        "role": "user",
        "content": f"Context:\n{poorly_curated_context}\n\nQuestion: {question}"
    }]
)
print("Answer:", response_poor.content[0].text)
print()

print("=== Well-curated context (only the relevant fact, clearly labelled) ===")
response_good = client.messages.create(
    model="claude-haiku-4-5",
    max_tokens=60,
    messages=[{
        "role": "user",
        "content": f"Context:\n{well_curated_context}\n\nQuestion: {question}"
    }]
)
print("Answer:", response_good.content[0].text)
print()

print("Both versions likely produce a correct answer for this short example —")
print("but as context grows longer and noisier, the gap between these two")
print("approaches widens significantly. This is a small-scale illustration of")
print("a problem that becomes much more serious with longer, messier real context.")

## My D365 analogy

Context engineering is similar to deciding exactly which documents to physically
bring into a client meeting room, in what order to present them, and which ones
to leave in the filing cabinet entirely — versus simply wordsmithing what you plan
to SAY once you are in the room.

Prompt engineering is the wording of your actual talking points. Context engineering
is the separate decision of which contracts, which prior correspondence, which
configuration printouts you are bringing into the room at all, and the order you
lay them out on the table.

Bringing every document that might conceivably be relevant "just in case" does not
make the meeting go better. It buries the one document that actually matters under
a pile of mostly-irrelevant paperwork, and makes it harder for everyone in the room —
including you — to find and weight the one fact that actually decides the outcome.

A senior consultant does not bring every file. They decide, deliberately, exactly
what belongs on the table for THIS specific conversation, and in what order it
should be encountered. That deliberate curation is the actual skill — separate
from, and just as important as, how eloquently you speak once you are in the room.

## LinkedIn post idea

A perfectly worded instruction to an AI model can still produce a bad answer —
not because the wording was wrong, but because of what was sitting around it
in the model's context window.

This is the difference between prompt engineering and context engineering.
Prompt engineering is HOW you word the instruction. Context engineering is WHAT
information you put in front of the model in the first place, from which sources,
in what order, and how much of each.

It is easy to assume that giving an AI system MORE background information always
helps. It does not. Irrelevant or poorly arranged context can bury the one fact
that actually matters, and the model has no way to know that fact deserved more
attention than the five paragraphs surrounding it.

I have sat in enough client meetings to know this lesson already, just under a
different name. Bringing every possible document into the room does not make the
conversation go better. It buries the one document that actually decides the outcome.

The skill was never "talk well once you are in the room." It was deciding, deliberately,
what belongs on the table in the first place.